# 🌟 MangaTranslator — Colab Edition
**Features:** API key rotation • Model fallback chain • Auto-retry failed pages • CBZ export • Resume-friendly

> **Runtime:** `Runtime → Change runtime type → T4 GPU`

### Two modes:
- **Mode A (Cells 1–6):** CLI batch — upload images, translate, download as CBZ/ZIP
- **Mode B (Cells 1–3, then 7):** Web UI — full Gradio interface with public share link

Run cells 1–3 first (setup), then choose your mode.

In [ ]:
# 1️⃣ Clone & Install
!git clone https://github.com/harami-boi/MangaTranslator.git 2>/dev/null; echo 'Repo ready'
%cd MangaTranslator
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade -q
!pip install -r requirements.txt -q
print('✅ All dependencies installed')

In [ ]:
# 2️⃣ Download Fonts
!mkdir -p fonts/Komika "fonts/Anime Ace"
!wget -q -O komika.zip "https://dl.dafont.com/dl/?f=komika_axis"
!wget -q -O animeace.zip "https://dl.dafont.com/dl/?f=anime_ace"
!unzip -q -j -o komika.zip "*.ttf" "*.otf" "*.TTF" "*.OTF" -d fonts/Komika/
!unzip -q -j -o animeace.zip "*.ttf" "*.otf" "*.TTF" "*.OTF" -d "fonts/Anime Ace/"
!rm -f komika.zip animeace.zip
!rm -rf fonts/.ipynb_checkpoints
print('✅ Fonts ready')

In [ ]:
# 3️⃣ Configuration — Used by BOTH Web UI and CLI batch modes

# === API KEYS (add as many as you want — rotation is automatic) ===
API_KEYS = [
    # Paste your Google AI Studio keys here, one per line:
    # "AIzaSy...",
    # "AIzaSy...",
]

# === MODEL FALLBACK CHAIN ===
# Models tried in order. When one fails or all keys exhaust, falls to next.
MODEL_CHAIN = [
    "gemini-2.5-flash-lite",     # Fastest, cheapest — best for bulk
    "gemini-2.0-flash-lite",     # Fallback lite
    "gemini-2.5-flash",          # More capable fallback
    "gemini-2.0-flash",          # Last resort
]

# === TRANSLATION SETTINGS ===
INPUT_LANGUAGE = "Japanese"
OUTPUT_LANGUAGE = "English"
ENABLE_OSB = True              # Outside speech bubble text detection
MAX_RETRIES_PER_IMAGE = 3      # Retry failed images this many times
RETRY_FAILED_AT_END = True     # Re-attempt all failed pages after first pass

print(f'✅ Config: {len(API_KEYS)} keys, {len(MODEL_CHAIN)} models in fallback chain')
print(f'   Models: {", ".join(MODEL_CHAIN)}')
print(f'   {INPUT_LANGUAGE} → {OUTPUT_LANGUAGE} | OSB: {"ON" if ENABLE_OSB else "OFF"}')

In [ ]:
# 4️⃣ Upload manga images (ZIP or individual files) — CLI BATCH MODE
import os, shutil, zipfile, tempfile
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('./input_manga')
OUTPUT_DIR = Path('./output')
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print('Upload your manga images or a ZIP file:')
uploaded = files.upload()

for fname, data in uploaded.items():
    if fname.lower().endswith('.zip'):
        tmp = Path(tempfile.mktemp(suffix='.zip'))
        tmp.write_bytes(data)
        with zipfile.ZipFile(tmp) as zf:
            zf.extractall(INPUT_DIR)
        tmp.unlink()
        print(f'  📦 Extracted ZIP: {fname}')
    else:
        (INPUT_DIR / fname).write_bytes(data)
        print(f'  🖼️ Uploaded: {fname}')

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
all_images = sorted([f for f in INPUT_DIR.rglob('*') if f.suffix.lower() in IMAGE_EXTS])
print(f'\n✅ Found {len(all_images)} images to translate')

In [ ]:
# 5️⃣ Batch Translate — Key Rotation + Model Fallback + Auto-Retry
import time, re, sys
import torch
from pathlib import Path
from core.config import (
    CleaningConfig, DetectionConfig, MangaTranslatorConfig,
    OutputConfig, OutsideTextConfig, PreprocessingConfig,
    RenderingConfig, TranslationConfig,
)
from core.pipeline import translate_and_render
from core.validation import autodetect_yolo_model_path, clamp_settings

models_dir = Path('./models').resolve()
models_dir.mkdir(parents=True, exist_ok=True)
yolo_path = str(autodetect_yolo_model_path(models_dir, 'yolo_1'))
font_dir = './fonts'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')

def build_config(api_key, model_name):
    config = MangaTranslatorConfig(
        yolo_model_path=yolo_path, verbose=False, device=device, parallel_requests=1,
        detection=DetectionConfig(
            confidence=0.6, conjoined_confidence=0.35, panel_confidence=0.25,
            seg_model='yolo', bubble_detector_model='yolo_1',
            conjoined_detection=True, use_panel_sorting=True, use_osb_text_verification=True,
        ),
        cleaning=CleaningConfig(thresholding_value=190, roi_shrink_px=5),
        translation=TranslationConfig(
            provider='Google', google_api_key=api_key, model_name=model_name,
            temperature=0.1, top_p=0.95, top_k=64,
            input_language=INPUT_LANGUAGE, output_language=OUTPUT_LANGUAGE,
            reading_direction='rtl', translation_mode='one-step',
            reasoning_effort='none', send_full_page_context=True,
            upscale_method='lanczos', context_image_max_side_pixels=768,
            bubble_min_side_pixels=128, ocr_method='LLM',
        ),
        rendering=RenderingConfig(font_dir=font_dir, max_font_size=16, min_font_size=8, supersampling_factor=2),
        output=OutputConfig(output_format='jpeg', jpeg_quality=92),
        outside_text=OutsideTextConfig(enabled=ENABLE_OSB, inpainting_method='opencv', osb_confidence=0.6, osb_outline_width=3.0),
        preprocessing=PreprocessingConfig(enabled=False, auto_scale=True),
    )
    clamp_settings(config)
    return config

RATE_LIMIT_KW = ['429', 'Rate limit', 'RESOURCE_EXHAUSTED', 'Resource Exhausted', 'quota']
MODEL_ERR_KW = ['not found', 'does not exist', 'is not available', 'unsupported model', 'invalid model']

# ── Rotation state ──
key_idx = 0
model_idx = 0
exhausted_keys = set()
dead_models = set()
failed_images = []
success = 0
skipped = 0
times = []

def get_key():
    global key_idx
    if not API_KEYS: return None
    for _ in range(len(API_KEYS)):
        if (key_idx % len(API_KEYS)) not in exhausted_keys:
            return API_KEYS[key_idx % len(API_KEYS)]
        key_idx += 1
    return None

def get_model():
    for i in range(model_idx, len(MODEL_CHAIN)):
        if MODEL_CHAIN[i] not in dead_models: return MODEL_CHAIN[i]
    return None

def advance_key():
    global key_idx
    exhausted_keys.add(key_idx % len(API_KEYS))
    key_idx += 1

def advance_model():
    global model_idx, exhausted_keys
    m = get_model()
    if m: dead_models.add(m)
    model_idx += 1
    exhausted_keys.clear()

def translate_one(img_path, out_path):
    global key_idx
    for _ in range(MAX_RETRIES_PER_IMAGE * len(MODEL_CHAIN)):
        model = get_model()
        key = get_key()
        if not model: return False, 'All models exhausted'
        if not key:
            old = model
            advance_model()
            model = get_model()
            if not model: return False, f'All keys exhausted for {old}, no more models'
            key = get_key()
            if not key: return False, 'No available keys'
            print(f'\n       ↳ Fallback: {old} → {model}')
        try:
            translate_and_render(img_path, build_config(key, model), out_path)
            return True, None
        except Exception as e:
            err = str(e)
            if any(kw in err for kw in RATE_LIMIT_KW):
                print(f' ⚠️ Key#{(key_idx%len(API_KEYS))+1} limited', end='', flush=True)
                advance_key()
            elif any(kw.lower() in err.lower() for kw in MODEL_ERR_KW):
                print(f' ⚠️ {model} unavailable', end='', flush=True)
                advance_model()
            else:
                return False, err[:200]
    return False, 'Max retries exceeded'

# ── Main loop ──
images = sorted([f for f in INPUT_DIR.rglob('*') if f.suffix.lower() in IMAGE_EXTS])
total_start = time.time()
print(f'\n🚀 Translating {len(images)} images | {len(API_KEYS)} keys | Models: {", ".join(MODEL_CHAIN)}')
print('-' * 60)

for i, img in enumerate(images):
    rel = img.relative_to(INPUT_DIR)
    out = OUTPUT_DIR / rel.with_suffix('.jpg')
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        skipped += 1
        print(f'  [{i+1}/{len(images)}] ⏭️  SKIP: {rel}')
        continue
    eta = time.strftime('%M:%S', time.gmtime((sum(times)/len(times))*(len(images)-i))) if times else '...'
    m = get_model() or '?'
    kl = (key_idx % len(API_KEYS))+1 if API_KEYS else 0
    print(f'  [{i+1}/{len(images)}] 🔑#{kl} {m} | {rel} (ETA: {eta})', end='', flush=True)
    t0 = time.time()
    ok, err = translate_one(img, out)
    dt = time.time() - t0
    if ok:
        times.append(dt); success += 1; print(f' ✅ {dt:.1f}s')
    else:
        failed_images.append((img, out, err)); print(f' ❌ {dt:.1f}s — {err[:100]}')

# ── Retry pass ──
if RETRY_FAILED_AT_END and failed_images:
    print(f'\n🔁 RETRYING {len(failed_images)} FAILED IMAGES')
    exhausted_keys.clear(); dead_models.clear(); model_idx = 0; key_idx = 0
    still_failed = []
    for j, (img, out, _) in enumerate(failed_images):
        if out.exists(): continue
        rel = img.relative_to(INPUT_DIR)
        print(f'  [retry {j+1}/{len(failed_images)}] {get_model() or "?"} | {rel}', end='', flush=True)
        t0 = time.time()
        ok, err = translate_one(img, out)
        dt = time.time() - t0
        if ok:
            success += 1; times.append(dt); print(f' ✅ {dt:.1f}s')
        else:
            still_failed.append((img, out, err)); print(f' ❌ {dt:.1f}s')
    failed_images = still_failed

total_time = time.time() - total_start
print(f'\n{"="*60}\n📊 DONE: {success} ✅ | {skipped} ⏭️ | {len(failed_images)} ❌ | {time.strftime("%H:%M:%S", time.gmtime(total_time))}\n{"="*60}')

In [ ]:
# 6️⃣ Download — CBZ (manga reader) or ZIP
import zipfile, re, os
from pathlib import Path
from xml.etree.ElementTree import Element, SubElement, tostring
from google.colab import files

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
out_imgs = sorted(
    [f for f in OUTPUT_DIR.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXTS],
    key=lambda x: tuple(int(p) if p.isdigit() else p for p in re.split(r'(\d+)', x.stem))
)

if not out_imgs:
    print('❌ No translated images found!')
else:
    title = 'MangaTranslator_output'
    # ComicInfo.xml for manga readers
    root = Element('ComicInfo')
    root.set('xmlns:xsi', 'http://www.w3.org/2001/XMLSchema-instance')
    root.set('xmlns:xsd', 'http://www.w3.org/2001/XMLSchema')
    SubElement(root, 'Title').text = title
    SubElement(root, 'PageCount').text = str(len(out_imgs))
    SubElement(root, 'LanguageISO').text = 'en'
    SubElement(root, 'Manga').text = 'YesAndRightToLeft'
    comic_xml = b'<?xml version="1.0" encoding="utf-8"?>\n' + tostring(root, encoding='unicode').encode('utf-8')

    # CBZ
    cbz = f'/tmp/{title}.cbz'
    pad = max(3, len(str(len(out_imgs))))
    with zipfile.ZipFile(cbz, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('ComicInfo.xml', comic_xml)
        for i, img in enumerate(out_imgs, 1):
            zf.write(img, f'{str(i).zfill(pad)}{img.suffix.lower()}')

    # ZIP
    zp = f'/tmp/{title}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for img in out_imgs:
            zf.write(img, str(img.relative_to(OUTPUT_DIR)))

    print(f'📦 CBZ: {len(out_imgs)} pages ({os.path.getsize(cbz)/1024/1024:.1f} MB)')
    print(f'📁 ZIP: {os.path.getsize(zp)/1024/1024:.1f} MB')
    choice = input('\n[1] CBZ (manga readers)  [2] ZIP  [3] Both : ').strip()
    if choice in ('1','3'): files.download(cbz)
    if choice in ('2','3'): files.download(zp)
    if choice not in ('1','2','3'): files.download(cbz); print('(Defaulted to CBZ)')

---
## 🌐 Mode B: Web UI
Run cells **1–3** above first, then run cell **7** below.

This writes your API key + settings into the app's config, and launches the full Gradio Web UI with a **public URL** you can open on any device.

The Web UI includes the **Download All as CBZ/ZIP** buttons in the Batch tab.

In [ ]:
# 7️⃣ Launch Web UI with your API key pre-configured
import json, os
from pathlib import Path

# ── Write config.json so the Web UI loads with your settings ──
config_dir = Path.home() / 'MangaTranslator'
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / 'config.json'

# Use the first API key for the Web UI (it handles one key at a time)
primary_key = API_KEYS[0] if API_KEYS else ''
primary_model = MODEL_CHAIN[0] if MODEL_CHAIN else 'gemini-2.5-flash-lite'

config = {
    'provider': 'Google',
    'model_name': primary_model,
    'google_api_key': primary_key,
    'provider_models': {
        'Google': primary_model,
    },
    'input_language': INPUT_LANGUAGE,
    'output_language': OUTPUT_LANGUAGE,
    'batch_input_language': INPUT_LANGUAGE,
    'batch_output_language': OUTPUT_LANGUAGE,
    'reading_direction': 'rtl',
    'translation_mode': 'one-step',
    'reasoning_effort': None,
    'temperature': 0.1,
    'top_p': 0.95,
    'top_k': 64,
    'send_full_page_context': True,
    'upscale_method': 'lanczos',
    'outside_text_enabled': ENABLE_OSB,
    'outside_text_inpainting_method': 'opencv',
    'outside_text_osb_confidence': 0.6,
    'output_format': 'jpeg',
    'jpeg_quality': 92,
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'✅ Config written to {config_path}')
print(f'   Provider: Google | Model: {primary_model}')
print(f'   API Key:  {primary_key[:12]}...{primary_key[-4:]}' if primary_key else '   ⚠️  No API key set! Add keys to Cell 3.')
print(f'   {INPUT_LANGUAGE} → {OUTPUT_LANGUAGE} | OSB: {"ON" if ENABLE_OSB else "OFF"}')

print('\n🚀 Starting Web UI... wait for the public URL below \u2193')
print('=' * 60)
# We use --share to natively enable the Gradio public link without any sed hacks!
!python app.py --share